In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
dataset = pd.read_csv('customer_propensity_processed_data.csv')
X = dataset.drop("purchased", axis=1)
y = dataset["purchased"]

In [3]:
dataset.head()

,gender_Female,gender_Male,marital_status_Divorced,marital_status_Married,marital_status_Single,marital_status_Widowed,education_Bachelor,education_Diploma,education_High School,education_Master,...,days_since_last_purchase,returned_orders,purchased,purchase_frequency,avg_monthly_spend,cart_abandonment_rate,purchase_conversion_rate,browsing_to_purchase_ratio,rfm_score,age_group
0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,128.0,1.0,0,0.86,14567.65,0.038,1.333,2.52,14,30
1,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,164.0,0.0,0,0.18,109.88,0.600,0.143,29.00,4,20
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,74.0,5.0,1,1.20,5717.82,0.032,2.458,1.65,15,40
3,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,...,162.0,0.0,0,0.25,799.07,0.250,0.111,21.33,6,40
4,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,145.0,0.0,0,0.38,1189.47,0.091,0.760,5.70,9,40


In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

In [5]:
from sklearn.preprocessing import StandardScaler

numeric_features = [
    'age_group',
    'website_visits_last_30d',
    'app_sessions_last_30d',
    'avg_session_duration',
    'product_views',
    'categories_viewed',
    'search_count',
    'wishlist_items',
    'total_orders',
    'average_order_value',
    'total_amount_spent',
    'months_active',
    'days_since_last_purchase',
    'returned_orders',
    'discount_percent',
    'campaign_duration_days',
    'campaign_send_hour',
    'campaign_cost',
    'previous_campaigns_received',
    'previous_campaign_response_rate',
    'annual_income',
    'purchase_frequency',
    'purchase_conversion_rate',
    'browsing_to_purchase_ratio',
    'avg_monthly_spend',
    'cart_abandonments',
    'rfm_score'
]

scaler = StandardScaler()

X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

In [6]:
print(y_train.value_counts())

purchased
0    27247
1    10312
Name: count, dtype: int64


In [7]:
from imblearn.combine import SMOTEENN
smote = SMOTEENN(sampling_strategy = 0.6777, random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

C:\Users\Nafees\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\Nafees\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "C:\Users\Nafees\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Nafees\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^

In [8]:
print("After SMOTE:")
print(y_train_smote.value_counts())

After SMOTE:
purchased
1    14978
0    14832
Name: count, dtype: int64


In [9]:
from sklearn.ensemble import RandomForestClassifier
classifier = RandomForestClassifier(n_estimators = 100, criterion = 'entropy', random_state = 0)
classifier.fit(X_train_smote, y_train_smote)

RandomForestClassifier(criterion='entropy', random_state=0)

In [10]:
y_pred = classifier.predict(X_test)

In [11]:
from sklearn.metrics import confusion_matrix, accuracy_score
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

[[7073 1958]
 [ 463 3026]]


0.806629392971246

In [12]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.78      0.85      9031
           1       0.61      0.87      0.71      3489

    accuracy                           0.81     12520
   macro avg       0.77      0.83      0.78     12520
weighted avg       0.85      0.81      0.81     12520



In [13]:
y_test.value_counts()

purchased
0    9031
1    3489
Name: count, dtype: int64

In [14]:
importances = classifier.feature_importances_

In [15]:
df_importance = pd.Series(importances, index=X.columns).sort_values(ascending = False)

In [16]:
df_importance.head(50)

days_since_last_purchase           0.114783
total_orders                       0.110267
rfm_score                          0.096189
purchase_frequency                 0.086057
product_views                      0.067174
cart_abandonments                  0.052610
purchase_conversion_rate           0.047207
total_amount_spent                 0.038541
website_visits_last_30d            0.038438
app_sessions_last_30d              0.027098
previous_campaign_response_rate    0.021578
annual_income                      0.021240
avg_session_duration               0.017944
returned_orders                    0.017880
avg_monthly_spend                  0.017711
browsing_to_purchase_ratio         0.017558
cart_abandonment_rate              0.016066
months_active                      0.015374
search_count                       0.013678
age_group                          0.012935
previous_campaigns_received        0.012585
average_order_value                0.009374
discount_percent                

In [17]:
df_importance.tail(50)

occupation_Engineer                   0.001473
favorite_category_Fashion             0.001427
favorite_category_Groceries           0.001417
education_Diploma                     0.001411
campaign_type_New Arrival             0.001381
campaign_send_day_Wednesday           0.001372
favorite_category_Beauty              0.001371
campaign_send_day_Thursday            0.001359
favorite_category_Home Decor          0.001281
city_Ahmedabad                        0.001257
campaign_channel_Whatsapp             0.001248
campaign_type_Loyalty Reward          0.001248
campaign_send_day_Monday              0.001242
campaign_channel_Sms                  0.001234
occupation_Teacher                    0.001214
city_Pune                             0.001193
campaign_channel_Push Notification    0.001191
city_Delhi                            0.001178
occupation_Doctor                     0.001169
favorite_category_Electronics         0.001163
favorite_category_Gaming              0.001154
city_Kolkata 